# Paper Trading Recorder

This notebook creates the paper-trading recording layer for the current supported market-making candidate. The goal is not to send real orders yet. The goal is to produce the exact event stream we need before live risk: decision, replay projection, market snapshot, submit, ack, cancel, cancel ack, and fill records.

The output is an append-only JSONL event log with a hash chain. That gives us a reproducible audit trail and lets us compare paper/live behavior against the historical replay model.


## Event Model

Every paper order should produce lifecycle events:

$$
Decision \rightarrow Submit \rightarrow Ack \rightarrow (Fill \;\text{or}\; CancelRequest \rightarrow CancelAck)
$$

The minimum comparison we need is:

$$
\Delta fill = I^{paper}_{fill} - I^{replay}_{fill}
$$

$$
\Delta q = q^{paper}_{filled} - q^{replay}_{filled}
$$

$$
\Delta t = t^{paper}_{ack/fill/cancel} - t^{model}_{ack/fill/cancel}
$$

A paper run is useful only if it records enough timing and market-state detail to explain those gaps.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import sys
from pathlib import Path

import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display


def find_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "notebooks" / "quote_policy_optimizer.ipynb").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing notebooks/quote_policy_optimizer.ipynb")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

from paper_trade_recorder import (  # noqa: E402
    PaperTradeRecorder,
    build_order_plan,
    lifecycle_summary,
    read_event_log,
    record_replay_projection,
    replay_comparison,
    select_replay_orders,
    validate_event_log,
)

QUOTE_POLICY_NOTEBOOK = PROJECT_ROOT / "notebooks" / "quote_policy_optimizer.ipynb"
FEATURE_ROOT = Path(os.environ.get("MODL_WS_FEATURE_DIR", "/mnt/burner-archive/ws_features")).expanduser()
BAR_MINUTES = int(os.environ.get("MODL_BAR_MINUTES", "5"))

RUN_ID = os.environ.get("MODL_PAPER_RUN_ID", pd.Timestamp.utcnow().strftime("paper-%Y%m%dT%H%M%SZ"))
PAPER_OUTPUT_ROOT = Path(os.environ.get("MODL_PAPER_OUTPUT_ROOT", "/tmp/modl_paper_trading_recorder")).expanduser()
ORDER_QTY = float(os.environ.get("MODL_STRATEGY_REPLAY_ORDER_QTY", "0.01"))
MAX_DRY_RUN_ORDERS = int(os.environ.get("MODL_PAPER_MAX_DRY_RUN_ORDERS", "40"))
ACK_LATENCY_MS = int(os.environ.get("MODL_PAPER_ACK_LATENCY_MS", "50"))
CANCEL_ACK_LATENCY_MS = int(os.environ.get("MODL_PAPER_CANCEL_ACK_LATENCY_MS", "100"))
RESET_DRY_RUN_OUTPUT = os.environ.get("MODL_PAPER_RESET_DRY_RUN_OUTPUT", "1") == "1"
SAVE_OUTPUTS = False

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 260)
pd.set_option("display.max_colwidth", 240)

PROJECT_ROOT, RUN_ID, PAPER_OUTPUT_ROOT, ORDER_QTY


## Load Current Quote Policy

We execute `quote_policy_optimizer.ipynb` and reuse its `paper_candidates`, `policy_frontier`, and `strategy_replay_by_ttl` tables. This keeps the paper recorder aligned with the latest research model instead of copying numbers by hand.


In [ ]:
def quiet_display(*args, **kwargs):
    return None


def execute_notebook(path: Path) -> dict[str, object]:
    original_display = ipd.display
    original_cwd = Path.cwd()
    ipd.display = quiet_display
    namespace: dict[str, object] = {"__name__": "__paper_source__"}
    notebook = json.loads(path.read_text())
    try:
        os.chdir(PROJECT_ROOT)
        for cell in notebook["cells"]:
            if cell.get("cell_type") != "code":
                continue
            source = "".join(cell.get("source", []))
            if not source.strip():
                continue
            exec(compile(source, f"{path}:{cell.get('id', 'cell')}", "exec"), namespace)
            plt.close("all")
    finally:
        os.chdir(original_cwd)
        ipd.display = original_display
    return namespace


policy_ns = execute_notebook(QUOTE_POLICY_NOTEBOOK)
policy_frontier = policy_ns["policy_frontier"].copy()
paper_candidates = policy_ns["paper_candidates"].copy()
live_candidates = policy_ns["live_candidates"].copy()
strategy_replay_by_ttl = policy_ns["strategy_replay_by_ttl"].copy()
strategy_ns = policy_ns["strategy_ns"]
frontier_cols = policy_ns["frontier_cols"]

display(policy_ns["gate_summary"])
display(paper_candidates[frontier_cols].head(10))


## Select Paper Candidate

For now, we select the top paper-gate row by implementation-adjusted score. This is the candidate we want to paper trade first. If no paper row exists, this notebook stops rather than silently manufacturing an order set.


In [ ]:
if paper_candidates.empty:
    raise RuntimeError("No paper candidates available from quote_policy_optimizer.ipynb")

selected_candidate = paper_candidates.sort_values("implementation_score_bps", ascending=False).iloc[0].to_dict()
selected_policy = str(selected_candidate["policy"])
selected_latency_ms = int(selected_candidate["latency_ms"])
selected_ttl_ms = int(selected_candidate["ttl_ms"])
selected_rebate_bps = float(selected_candidate["maker_rebate_bps"])

selected_replay_orders = select_replay_orders(
    strategy_replay_by_ttl,
    selected_policy,
    selected_latency_ms,
    selected_ttl_ms,
    selected_rebate_bps,
    max_orders=MAX_DRY_RUN_ORDERS,
)
detailed_replay_source = "strategy_replay_by_ttl"
if selected_replay_orders.empty:
    selected_replay_orders = strategy_ns["replay_policy"](
        selected_policy,
        strategy_ns["policy_inputs"][selected_policy],
        selected_latency_ms,
        selected_ttl_ms,
        selected_rebate_bps,
    ).sort_values(["decision_mts", "side", "quote_distance_bps"]).head(MAX_DRY_RUN_ORDERS).reset_index(drop=True)
    detailed_replay_source = "strategy_replay.replay_policy"

if selected_replay_orders.empty:
    raise RuntimeError("Selected paper candidate produced no detailed replay orders")
order_plan = build_order_plan(selected_replay_orders, RUN_ID, ORDER_QTY)

candidate_summary = pd.DataFrame([selected_candidate])
plan_summary = pd.DataFrame(
    [
        {
            "run_id": RUN_ID,
            "policy": selected_policy,
            "latency_ms": selected_latency_ms,
            "ttl_ms": selected_ttl_ms,
            "maker_rebate_bps": selected_rebate_bps,
            "planned_orders": len(order_plan),
            "detailed_replay_source": detailed_replay_source,
            "expected_full_fills": int(order_plan["event_full_fill"].sum()) if len(order_plan) else 0,
            "expected_l2_coverage_rate": float(order_plan["l2_price_covered"].mean()) if len(order_plan) else np.nan,
            "expected_ev_after_rebate_bps": float(order_plan["event_value_after_rebate_bps"].mean()) if len(order_plan) else np.nan,
        }
    ]
)

display(candidate_summary[frontier_cols])
display(plan_summary)
display(order_plan.head(20))


## Write A Dry-Run Paper Event Log

This dry run writes to `/tmp` by default. In production paper trading, the same recorder should write to a persistent directory, for example `/mnt/burner-archive/paper_trading`.

The dry run uses historical replay rows as the event source so we can validate schema and lifecycle accounting before wiring a live exchange API.


In [ ]:
if RESET_DRY_RUN_OUTPUT and PAPER_OUTPUT_ROOT.exists():
    shutil.rmtree(PAPER_OUTPUT_ROOT)

recorder = PaperTradeRecorder(
    PAPER_OUTPUT_ROOT,
    RUN_ID,
    venue="hibachi",
    symbol="BTC/USDT-P",
    strategy=selected_policy,
)

paper_events = record_replay_projection(
    recorder,
    order_plan,
    ack_latency_ms=ACK_LATENCY_MS,
    cancel_ack_latency_ms=CANCEL_ACK_LATENCY_MS,
)

paper_event_path = recorder.path
paper_event_path, len(paper_events)


## Validate Log Integrity

The log is useful only if it is append-only and replayable. These checks verify required columns, event IDs, event types, monotonic event timestamps, and the hash chain.


In [ ]:
paper_events = read_event_log(paper_event_path)
validation = validate_event_log(paper_events)
lifecycle = lifecycle_summary(paper_events)

display(validation)
display(lifecycle.head(20))

if not validation["passed"].all():
    raise RuntimeError("Paper event log validation failed")


## Compare Paper Log To Replay

This table is the first version of paper-vs-replay reconciliation. In a real paper run, the paper fields will come from actual exchange acknowledgements, cancels, and fills. The same checks should then show where live mechanics diverge from simulation.


In [ ]:
paper_replay_comparison = replay_comparison(order_plan, lifecycle)
comparison_summary = pd.DataFrame(
    [
        {
            "orders": len(paper_replay_comparison),
            "expected_full_fills": int(paper_replay_comparison["event_full_fill"].sum()) if len(paper_replay_comparison) else 0,
            "recorded_full_fills": int(paper_replay_comparison["recorded_full_fill"].sum()) if len(paper_replay_comparison) else 0,
            "fill_match_rate": float(paper_replay_comparison["fill_match"].mean()) if len(paper_replay_comparison) else np.nan,
            "mean_qty_gap": float(paper_replay_comparison["qty_gap"].mean()) if len(paper_replay_comparison) else np.nan,
            "median_submit_to_ack_ms": float(lifecycle["submit_to_ack_ms"].median()) if len(lifecycle) else np.nan,
            "median_cancel_latency_ms": float(lifecycle["cancel_latency_ms"].median()) if len(lifecycle) else np.nan,
        }
    ]
)

display(comparison_summary)
display(paper_replay_comparison.head(30))


## Recorder Diagnostics

The plots show timing and lifecycle state. For a real paper run, these should be watched daily: ack latency, cancel latency, fill delay, and mismatch to replay.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

sns.countplot(data=paper_events, x="event_type", ax=axes[0, 0])
axes[0, 0].tick_params(axis="x", rotation=45)
axes[0, 0].set_title("Paper Event Counts")

sns.countplot(data=lifecycle, x="status", ax=axes[0, 1])
axes[0, 1].set_title("Order Lifecycle Status")

latency_frame = lifecycle[["submit_to_ack_ms", "cancel_latency_ms", "submit_to_fill_ms"]].melt(
    var_name="latency_type",
    value_name="latency_ms",
).dropna()
sns.histplot(data=latency_frame, x="latency_ms", hue="latency_type", bins=30, ax=axes[1, 0])
axes[1, 0].set_title("Paper Timing Distribution")

sns.scatterplot(
    data=paper_replay_comparison,
    x="l2_queue_ahead_qty",
    y="event_value_after_rebate_bps",
    hue="recorded_full_fill",
    style="l2_price_covered",
    ax=axes[1, 1],
)
axes[1, 1].axhline(0, color="black", linewidth=1)
axes[1, 1].set_title("Replay EV Versus L2 Queue For Recorded Orders")

plt.tight_layout()


## Paper Trading Recorder Checklist

Before this becomes a live paper process, the order manager must write the same event schema from actual runtime events:

1. `decision`: model score, selected side, distance, price, quantity, inventory, support source, feature timestamp.
2. `market_snapshot`: best bid/ask, L2 snapshot timestamp, queue-ahead estimate, snapshot age, sequence if available.
3. `order_submit`: local send timestamp, post-only flag, client order ID, price, size, TTL.
4. `order_ack`: venue order ID, ack timestamp, ack latency, accepted/rejected status.
5. `order_fill`: fill timestamp, fill price, fill size, maker/taker flag, fee/rebate, remaining quantity.
6. `order_cancel_request`: local cancel timestamp and reason.
7. `order_cancel_ack`: cancel ack timestamp and cancel latency.
8. `heartbeat`: strategy process health, market-data freshness, open order count, position/inventory.

The first paper-trading objective is not PnL. It is simulator validation: do paper fills, queue behavior, and markouts look like the replay notebook predicted?


## Optional Save

Set `SAVE_OUTPUTS = True` to copy the dry-run event log and write summary tables under `MODL_WS_FEATURE_DIR/paper_trading_recorder/bar_<N>m/run_id=<RUN_ID>`.


In [ ]:
if SAVE_OUTPUTS:
    out_dir = FEATURE_ROOT / "paper_trading_recorder" / f"bar_{BAR_MINUTES}m" / f"run_id={RUN_ID}"
    out_dir.mkdir(parents=True, exist_ok=True)
    order_plan.to_parquet(out_dir / "order_plan.parquet")
    paper_events.to_parquet(out_dir / "paper_events.parquet")
    lifecycle.to_parquet(out_dir / "lifecycle.parquet")
    validation.to_parquet(out_dir / "validation.parquet")
    paper_replay_comparison.to_parquet(out_dir / "paper_replay_comparison.parquet")
    comparison_summary.to_parquet(out_dir / "comparison_summary.parquet")
    shutil.copy2(paper_event_path, out_dir / "paper_events.jsonl")
    print(f"wrote paper trading recorder outputs to {out_dir}")
else:
    print("SAVE_OUTPUTS is false; dry-run JSONL remains under", paper_event_path)
